In [2]:
import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import KFold
import polars as pl
import os
from typing import Optional

## General Testing

In [82]:
class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_dim_1, num_layers, output_size, hidden_dim_2: int = 32, dropout_rate: float = 0.3):
        super(LSTMClassifier, self).__init__()
        self.hidden_dim_1 = hidden_dim_1
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size,
                            self.hidden_dim_1,
                            num_layers=num_layers,
                            batch_first=True
                           )
        #self.dropout_1 = nn.Dropout(dropout_rate)
        self.fc1 = nn.Linear(self.hidden_dim_1, hidden_dim_2)
        self.relu = nn.ReLU()
        #self.dropout_2 = nn.Dropout(dropout_rate)
        self.fc2 = nn.Linear(hidden_dim_2, output_size)
        #self.batch_size = 1
    
    def forward(self, x: torch.Tensor):
        h = torch.zeros(self.num_layers, x.size(0), self.hidden_dim_1).to(x.device)
        c = torch.zeros(self.num_layers, x.size(0), self.hidden_dim_1).to(x.device)
        # h = torch.zeros(self.num_layers, self.batch_size, self.hidden_dim).to(x.device)
        # c = torch.zeros(self.num_layers, self.batch_size, self.hidden_dim).to(x.device)
        out, _ = self.lstm(x, (h, c))
        out = out[:, -1, :]
        #out = self.dropout_1(out)
        out = self.fc1(out)
        out = self.relu(out)
        #out = self.dropout_2(out)
        out = self.fc2(out)
        return out

In [4]:
# Generate sample data
X_train = torch.rand(100, 10, 50)  # 100 sequences of length 10 with 50 features each
y_train = torch.randint(3, size=(100,))  # Binary labels (0 or 1)

print(X_train.shape)
y_train

torch.Size([100, 10, 50])


tensor([1, 1, 1, 0, 1, 1, 2, 0, 2, 1, 2, 0, 0, 2, 1, 1, 1, 0, 2, 2, 1, 2, 0, 0,
        1, 2, 2, 2, 2, 1, 1, 0, 0, 1, 0, 2, 0, 0, 1, 2, 0, 1, 0, 1, 2, 1, 2, 0,
        1, 0, 1, 2, 2, 2, 1, 2, 0, 1, 1, 0, 1, 2, 2, 2, 0, 0, 0, 2, 0, 2, 2, 2,
        1, 0, 0, 0, 2, 2, 2, 0, 1, 1, 0, 2, 2, 1, 2, 2, 2, 1, 1, 1, 0, 2, 2, 2,
        2, 0, 1, 0])

In [5]:
y_train.unique(return_counts=True)

(tensor([0, 1, 2]), tensor([30, 32, 38]))

In [6]:
batch_size = 1
batches_X = X_train.split(batch_size)
batches_y = y_train.split(batch_size)
print(batches_X[-1].shape)
batches_y[-1].shape

torch.Size([1, 10, 50])


torch.Size([1])

In [57]:
input_size = 50
hidden_size = 64
num_layers = 1
output_size = 3

model = LSTMClassifier(input_size, hidden_size, num_layers, output_size)
model

LSTMClassifier(
  (lstm): LSTM(50, 64, batch_first=True)
  (dropout_1): Dropout(p=0.3, inplace=False)
  (fc1): Linear(in_features=64, out_features=32, bias=True)
  (dropout_2): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=32, out_features=3, bias=True)
)

In [58]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

num_epochs = 10
for epoch in range(num_epochs):
    for bind in range(len(batches_X)):
        batch_X = batches_X[bind]
        batch_y = batches_y[bind]
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        print(f'Epoch [{epoch+1}/{num_epochs}], Batch [{bind}/{len(batches_X)}], Loss: {loss.item()}')

Epoch [1/10], Batch [0/100], Loss: 1.0710283517837524
Epoch [1/10], Batch [1/100], Loss: 0.6671250462532043
Epoch [1/10], Batch [2/100], Loss: 0.3299598693847656
Epoch [1/10], Batch [3/100], Loss: 3.0616395473480225
Epoch [1/10], Batch [4/100], Loss: 0.14825040102005005
Epoch [1/10], Batch [5/100], Loss: 0.10798610746860504
Epoch [1/10], Batch [6/100], Loss: 4.8511528968811035
Epoch [1/10], Batch [7/100], Loss: 3.326906442642212
Epoch [1/10], Batch [8/100], Loss: 2.2599830627441406
Epoch [1/10], Batch [9/100], Loss: 0.4860837459564209
Epoch [1/10], Batch [10/100], Loss: 2.3978488445281982
Epoch [1/10], Batch [11/100], Loss: 1.178433895111084
Epoch [1/10], Batch [12/100], Loss: 1.0268296003341675
Epoch [1/10], Batch [13/100], Loss: 1.4037244319915771
Epoch [1/10], Batch [14/100], Loss: 1.2232884168624878
Epoch [1/10], Batch [15/100], Loss: 1.2940211296081543
Epoch [1/10], Batch [16/100], Loss: 1.0889898538589478
Epoch [1/10], Batch [17/100], Loss: 1.0496946573257446
Epoch [1/10], Batch 

In [9]:
X_test = torch.rand(1, 10, 50)  # Test data with 10 sequences
with torch.no_grad():
    predictions = model(X_test)
    predicted_labels = torch.argmax(predictions, dim=1)
    print("Predicted Labels:", predicted_labels)

Predicted Labels: tensor([1])


## Gestures Application

### Initial Inference Test

In [10]:
stationary_df = pl.read_csv("./imu_data/rosbag/data/data_clean/rosbag2_2023_03_02-05_19_45_data.csv")

stationary_df

timestamp,Imu0_linear_accleration_x,Imu0_linear_accleration_y,Imu0_linear_accleration_z,Imu0_angular_velocity_x,Imu0_angular_velocity_y,Imu0_angular_velocity_z,Imu0_orientation_x,Imu0_orientation_y,Imu0_orientation_z,Imu0_orientation_w,Imu1_linear_accleration_x,Imu1_linear_accleration_y,Imu1_linear_accleration_z,Imu1_angular_velocity_x,Imu1_angular_velocity_y,Imu1_angular_velocity_z,Imu1_orientation_x,Imu1_orientation_y,Imu1_orientation_z,Imu1_orientation_w,Imu2_linear_accleration_x,Imu2_linear_accleration_y,Imu2_linear_accleration_z,Imu2_angular_velocity_x,Imu2_angular_velocity_y,Imu2_angular_velocity_z,Imu2_orientation_x,Imu2_orientation_y,Imu2_orientation_z,Imu2_orientation_w,imu0_to_imu1_translation_x,imu0_to_imu1_translation_y,imu0_to_imu1_translation_z,imu0_to_imu1_rotation_x,imu0_to_imu1_rotation_y,imu0_to_imu1_rotation_z,imu0_to_imu1_rotation_w,imu0_to_imu2_translation_x,imu0_to_imu2_translation_y,imu0_to_imu2_translation_z,imu0_to_imu2_rotation_x,imu0_to_imu2_rotation_y,imu0_to_imu2_rotation_z,imu0_to_imu2_rotation_w
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1.6777e9,-2.23,5.5,7.67,0.1,0.03,0.02,0.404766,0.594505,0.582337,0.378962,-7.88,5.79,0.74,0.05,-0.14,0.02,0.627936,0.249794,-0.113059,0.728366,-6.82,6.77,0.47,0.14,0.1,0.03,-0.963904,-0.092533,0.238984,0.0722,1.213515,1.764268,-0.192974,0.0,0.0,0.0,1.0,-1.793537,1.102632,0.435806,0.0,0.0,0.0,1.0
1.6777e9,-2.25,5.48,7.62,0.1,0.02,0.0,0.404378,0.594716,0.58233,0.379056,-7.69,5.67,0.75,0.07,-0.15,0.02,0.627782,0.249863,-0.11573,0.728056,-6.58,6.69,0.49,0.15,0.09,0.02,-0.963974,-0.091538,0.238823,0.073063,1.216771,1.761606,-0.196756,0.0,0.0,0.0,1.0,-1.793829,1.101421,0.437663,0.0,0.0,0.0,1.0
1.6777e9,-2.23,5.49,7.62,0.09,0.03,-0.0,0.403894,0.594895,0.582438,0.379125,-7.73,5.83,0.81,0.09,-0.13,0.01,0.627773,0.250132,-0.118321,0.727555,-6.57,6.83,0.43,0.15,0.11,0.0,-0.964077,-0.090771,0.238422,0.073964,1.220155,1.758638,-0.20227,0.0,0.0,0.0,1.0,-1.793816,1.100421,0.440226,0.0,0.0,0.0,1.0
1.6777e9,-2.21,5.54,7.66,0.07,0.04,0.01,0.403315,0.594936,0.58278,0.37915,-7.96,5.94,0.85,0.05,-0.13,0.01,0.627413,0.250454,-0.120852,0.727338,-6.8,6.93,0.52,0.13,0.11,-0.0,-0.964196,-0.090045,0.237997,0.074675,1.225142,1.754657,-0.206659,0.0,0.0,0.0,1.0,-1.793338,1.099994,0.44323,0.0,0.0,0.0,1.0
1.6777e9,-2.22,5.58,7.62,0.1,0.01,0.02,0.403134,0.595003,0.582819,0.379179,-7.66,5.75,0.63,0.04,-0.15,0.03,0.62707,0.250522,-0.123368,0.727188,-6.54,6.68,0.47,0.13,0.09,0.03,-0.964291,-0.088982,0.237819,0.075287,1.229533,1.751352,-0.208602,0.0,0.0,0.0,1.0,-1.793432,1.099052,0.44518,0.0,0.0,0.0,1.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
1.6777e9,-0.27,6.35,7.25,0.08,-0.04,0.03,0.362113,0.616319,0.604091,0.352277,-6.78,6.24,3.06,0.04,-0.17,0.01,0.584119,0.175574,-0.393573,0.687807,-5.11,7.39,2.34,0.12,0.07,0.01,-0.95567,-0.086922,0.124516,0.252261,1.375108,1.636072,-0.234188,0.0,0.0,0.0,1.0,-1.73238,0.816213,0.97732,0.0,0.0,0.0,1.0
1.6777e9,-0.28,6.37,7.21,0.07,-0.03,0.02,0.362252,0.615857,0.604321,0.352547,-6.77,6.36,3.14,0.06,-0.16,0.0,0.582648,0.173818,-0.395264,0.688531,-5.05,7.59,2.36,0.14,0.08,0.01,-0.95549,-0.086226,0.124267,0.2533,1.376354,1.635949,-0.227637,0.0,0.0,0.0,1.0,-1.731606,0.814533,0.980091,0.0,0.0,0.0,1.0
1.6777e9,-0.29,6.45,7.2,0.1,-0.02,0.02,0.362445,0.615628,0.604389,0.352633,-6.78,6.38,3.06,0.06,-0.17,0.02,0.581107,0.171811,-0.396931,0.689377,-5.17,7.47,2.37,0.15,0.07,0.03,-0.955277,-0.0853,0.124199,0.254449,1.376511,1.636915,-0.219605,0.0,0.0,0.0,1.0,-1.731446,0.811867,0.982582,0.0,0.0,0.0,1.0


In [11]:
def get_column_combos(imus, dims, readings):
    columns = [f"Imu{im}_{read}_{dim}" for im in imus for read in readings for dim in dims]
    return columns

imus = [0]
dims = ["x", "y", "z"]
readings = ["linear_accleration", "angular_velocity"]

X_one = stationary_df.select(
    get_column_combos(imus, dims, readings)
).to_torch(dtype=pl.Float32)

X_one.shape

torch.Size([100, 6])

In [12]:
stationary_df_label = pl.read_csv("./imu_data/rosbag/data/label/rosbag2_2023_03_02-05_19_45_label.csv")

stationary_df_label

label
i64
2


In [13]:
y_one = stationary_df_label.with_columns(
    pl.col('label').replace(10, 7)
).to_torch()[0]

y_one

tensor([2])

In [14]:
classes = {
    'STATIC': 0,
    'SLIDE_UP': 1,
    'SLIDE_DOWN': 2,
    'SLIDE_LEFT': 3,
    'SLIDE_RIGHT': 4,
    'RELEASE': 5,
    'GRASP': 6,
    'NONE': 7 # originally 10
}

input_size = 6
hidden_size = 64
num_layers = 1
output_size = len(classes)

model = LSTMClassifier(input_size, hidden_size, num_layers, output_size)
model

LSTMClassifier(
  (lstm): LSTM(6, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=8, bias=True)
)

In [15]:
X_one.unsqueeze(0).shape

torch.Size([1, 100, 6])

In [16]:
with torch.no_grad():
    predictions = model(X_one.unsqueeze(0))
    predicted_labels = torch.argmax(predictions, dim=1)
    print("Predicted Labels:", predicted_labels)

Predicted Labels: tensor([7])


### Data Loading

In [83]:
class IMUDataSet(Dataset):
    def __init__(
            self,
            data_dir: str,
            labels_dir: str,
            class_mappings: list[tuple[int, int]] = [(10, 7)],
            mean: Optional[torch.Tensor] = None,
            std: Optional[torch.Tensor] = None,
            normalize: bool = False):
        super().__init__()
        self.data_dir = data_dir
        self.labels_dir = labels_dir
        self.data_paths = os.listdir(self.data_dir)
        self.label_paths = os.listdir(self.labels_dir)
        self.class_mappings = class_mappings
        self.mean = mean
        self.std = std
        self.normalize = normalize
        self.cache = {}
        self.imus = [0]
        self.dims = ["x", "y", "z"]
        self.readings = ["linear_accleration", "angular_velocity"]
        self.label_col = "label"
        self.epsilon = 1e-7

    def __len__(self):
        return len(self.data_paths)

    def __getitem__(self, idx):
        data, label = self.cache.get(idx, (None, None))
        if data is not None and label is not None:
            return data, label
        
        data_df = pl.read_csv(os.path.join(self.data_dir, self.data_paths[idx]))
        X_item = data_df.select(
            self.get_column_combos()
        ).to_torch(dtype=pl.Float32)
        
        label_df = pl.read_csv(os.path.join(self.labels_dir, self.label_paths[idx]))
        y_item = label_df.with_columns(
            (pl.col(self.label_col).replace(before, after) for before, after in self.class_mappings)
        ).to_torch()[0]

        # if self.min_val is not None and self.max_val is not None:
        #     X_item = (X_item - self.min_val) / (self.max_val - self.min_val + self.epsilon)
        if self.normalize and self.mean is not None and self.std is not None:
            X_item = (X_item - self.mean) / self.std

        # if X_item.shape[0] > 100:
        #     print(X_item.shape[0], y_item)

        try:
            self.cache[idx] = X_item, y_item
        except OSError:
            del self.cache[list(self.cache.keys())[0]]

        return X_item, y_item
    
    def get_column_combos(self):
        columns = [f"Imu{im}_{read}_{dim}" for im in self.imus for read in self.readings for dim in self.dims]
        return columns

In [18]:
a = torch.ones(5, 2)
b = torch.ones(4, 2)
c = torch.ones(3, 2)
pad_sequence([a, b, c], batch_first=True, padding_side="left")

tensor([[[1., 1.],
         [1., 1.],
         [1., 1.],
         [1., 1.],
         [1., 1.]],

        [[0., 0.],
         [1., 1.],
         [1., 1.],
         [1., 1.],
         [1., 1.]],

        [[0., 0.],
         [0., 0.],
         [1., 1.],
         [1., 1.],
         [1., 1.]]])

In [94]:
def get_norm_values(data_loader: DataLoader):
    X_list = [X[0] for X, _ in data_loader]
    X_tensor = torch.cat(X_list)
    return X_tensor.mean(dim=0), X_tensor.std(dim=0)

temp_dataset = IMUDataSet("./imu_data/rosbag/data/data_clean", "./imu_data/rosbag/data/label")

temp_loader = torch.utils.data.DataLoader(temp_dataset,
        batch_size=1,
        shuffle=False
    )

mean, std = get_norm_values(temp_loader)
print("Mean:", mean)
print("STD:", std)

Mean: tensor([-0.7395,  4.8429,  3.3637,  0.0850,  0.0463,  0.0117])
STD: tensor([5.7835, 3.3225, 4.3546, 0.7242, 0.9477, 0.5873])


In [20]:
torch.ones(2, 5, 3) / torch.tensor([2, 3, 4])

tensor([[[0.5000, 0.3333, 0.2500],
         [0.5000, 0.3333, 0.2500],
         [0.5000, 0.3333, 0.2500],
         [0.5000, 0.3333, 0.2500],
         [0.5000, 0.3333, 0.2500]],

        [[0.5000, 0.3333, 0.2500],
         [0.5000, 0.3333, 0.2500],
         [0.5000, 0.3333, 0.2500],
         [0.5000, 0.3333, 0.2500],
         [0.5000, 0.3333, 0.2500]]])

In [114]:
def pad_collate(batch):
    tensors, targets = zip(*batch)
    data = pad_sequence(tensors, batch_first=True, padding_side="left")
    labels = torch.cat(targets)
    return data, labels

batch_size = 1
# min_val = torch.Tensor([-19.61, -19.61, -19.61, -4.36, -4.36, -4.36])
# max_val = torch.Tensor([19.61, 19.61, 19.61, 4.36, 4.36, 4.36])

full_dataset = IMUDataSet(
    "./imu_data/rosbag/data/data_clean", 
    "./imu_data/rosbag/data/label",
    mean=mean,
    std=std,
    normalize=True
)
#train_set, val_set = random_split(full_dataset, [0.95, 0.05])

# train_loader = torch.utils.data.DataLoader(train_set,
#     batch_size=batch_size,
#     collate_fn=pad_collate,
#     shuffle=True
# )

### Training

In [124]:
def reset_weights(m):
  '''
    Try resetting model weights to avoid
    weight leakage.
  '''
  for layer in m.children():
    if hasattr(layer, 'reset_parameters'):
        print(f'Reset trainable parameters of layer = {layer}')
        layer.reset_parameters()

classes = {
    'STATIC': 0,
    'SLIDE_UP': 1,
    'SLIDE_DOWN': 2,
    'SLIDE_LEFT': 3,
    'SLIDE_RIGHT': 4,
    'RELEASE': 5,
    'GRASP': 6,
    'NONE': 7 # originally 10
}

input_size = 6
hidden_size = 24
num_layers = 1
output_size = len(classes)

# model = LSTMClassifier(input_size, hidden_size, num_layers, output_size)
# model

In [ ]:
k_folds = 5
# num_epochs = 200
# learning_rate = 0.0005
num_epochs = 75
learning_rate = 0.001
weight_decay = 0.0001

results = {}

criterion = nn.CrossEntropyLoss()
#optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# for epoch in range(num_epochs):
#     for X, y in train_loader:
#         y_hat = model(X)
#         loss = criterion(y_hat, y)
#         optimizer.zero_grad()
#         loss.backward()
#         optimizer.step()
#     print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item()}')

kfold = KFold(n_splits=k_folds, shuffle=True)

# Start print
print('--------------------------------')

# K-fold Cross Validation model evaluation
for fold, (train_ids, test_ids) in enumerate(kfold.split(full_dataset)):

    # Print
    print(f'FOLD {fold}')
    print('--------------------------------')

    # Sample elements randomly from a given list of ids, no replacement.
    train_subsampler = torch.utils.data.SubsetRandomSampler(train_ids)
    test_subsampler = torch.utils.data.SubsetRandomSampler(test_ids)

    # Define data loaders for training and testing data in this fold
    trainloader = torch.utils.data.DataLoader(
                        full_dataset,
                        batch_size=batch_size,
                        collate_fn=pad_collate,
                        sampler=train_subsampler
    )
    testloader = torch.utils.data.DataLoader(
                        full_dataset,
                        batch_size=batch_size,
                        collate_fn=pad_collate,
                        sampler=test_subsampler
    )

    # Init the neural network
    model = LSTMClassifier(input_size, hidden_size, num_layers, output_size)
    model.apply(reset_weights)

    # Initialize optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    # Run the training loop for defined number of epochs
    for epoch in range(0, num_epochs):

        # Print epoch
        print(f'Starting epoch {epoch+1}')

        # Set current loss value
        current_loss = 0.0

        # Iterate over the DataLoader for training data
        for i, data in enumerate(trainloader, 0):
        
            # Get inputs
            inputs, targets = data
            
            # Zero the gradients
            optimizer.zero_grad()
            
            # Perform forward pass
            outputs = model(inputs)
            
            # Compute loss
            loss = criterion(outputs, targets)
            
            # Perform backward pass
            loss.backward()
            
            # Perform optimization
            optimizer.step()
            
            # Print statistics
            current_loss += loss.item()
            if i % 500 == 499:
                print('Loss after mini-batch %5d: %.3f' %
                        (i + 1, current_loss / 500))
                current_loss = 0.0
            
    # Process is complete.
    print('Training process has finished. Saving trained model.')

    # Print about testing
    print('Starting testing')

    # Saving the model
    save_path = f'./model/model-fold-{fold}.pth'
    torch.save(model.state_dict(), save_path)

    # Evaluationfor this fold
    correct, total = 0, 0
    with torch.no_grad():

        # Iterate over the test data and generate predictions
        for i, data in enumerate(testloader, 0):

            # Get inputs
            inputs, targets = data

            # Generate outputs
            outputs = model(inputs)

            # Set total and correct
            _, predicted = torch.max(outputs.data, 1)
            total += targets.size(0)
            correct += (predicted == targets).sum().item()

        # Print accuracy
        print('Accuracy for fold %d: %d %%' % (fold, 100.0 * correct / total))
        print('--------------------------------')
        results[fold] = 100.0 * (correct / total)

--------------------------------
FOLD 0
--------------------------------


ValueError: sampler option is mutually exclusive with shuffle

In [128]:
# val_loader = torch.utils.data.DataLoader(val_set,
#         batch_size=len(val_set),
#         collate_fn=pad_collate,
#         shuffle=False
#     )

# # validation data
# val_X, val_y = next(iter(val_loader))
# Y_hat = model(val_X)
# val_loss = criterion(Y_hat, val_y)
# val_err = val_loss.item()

# print("Validation loss:", val_err)
fold = 2
save_path = f'./model/model-fold-{fold}.pth'
state_dict = torch.load(save_path, weights_only=True)
model = LSTMClassifier(input_size, hidden_size, num_layers, output_size)
model.load_state_dict(state_dict)
model

LSTMClassifier(
  (lstm): LSTM(6, 24, batch_first=True)
  (fc1): Linear(in_features=24, out_features=32, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=32, out_features=8, bias=True)
)

In [108]:
print(X_one.shape)
y_one

torch.Size([100, 6])


tensor([2])

In [129]:
#to_pred = ((X_one - min_val) / (max_val - min_val + 1e-7)).unsqueeze(0)
to_pred = ((X_one - mean) / std).unsqueeze(0)
#to_pred = X_one.unsqueeze(0)
with torch.no_grad():
    predictions = model(to_pred)
    print(predictions)
    predicted_labels = torch.argmax(predictions, dim=1)
    print("Predicted Labels:", predicted_labels)

tensor([[-0.3573, -0.4141,  0.0954, -1.2628, -0.0863, -0.5164, -0.3307, -0.3323]])
Predicted Labels: tensor([2])


In [130]:
X_another = pl.read_csv("./imu_data/rosbag/data/data_clean/rosbag2_2023_02_10-07_52_42_data.csv").select(
    get_column_combos(imus, dims, readings)
).to_torch(dtype=pl.Float32)

y_another = pl.read_csv("./imu_data/rosbag/data/label/rosbag2_2023_02_10-07_52_42_label.csv").with_columns(
    pl.col('label').replace(10, 7)
).to_torch()[0]

print(X_another.shape)
y_another

torch.Size([100, 6])


tensor([1])

In [131]:
#to_pred = ((X_another - min_val) / (max_val - min_val + 1e-7)).unsqueeze(0)
to_pred = ((X_another - mean) / std).unsqueeze(0)
#to_pred = X_another.unsqueeze(0)
with torch.no_grad():
    predictions = model(to_pred)
    print(predictions)
    predicted_labels = torch.argmax(predictions, dim=1)
    print("Predicted Labels:", predicted_labels)

tensor([[-1.4862, -1.0406, -0.4007, -0.2760, -0.2000, -0.4082, -0.2093,  0.0959]])
Predicted Labels: tensor([7])


In [132]:
X_another = pl.read_csv("./imu_data/rosbag/data/data_clean/rosbag2_2023_02_27-13_52_46_data.csv").select(
    get_column_combos(imus, dims, readings)
).to_torch(dtype=pl.Float32)

y_another = pl.read_csv("./imu_data/rosbag/data/label/rosbag2_2023_02_27-13_52_46_label.csv").with_columns(
    pl.col('label').replace(10, 7)
).to_torch()[0]

print(X_another.shape)
y_another

torch.Size([184, 6])


tensor([7])

In [133]:
#to_pred = ((X_another - min_val) / (max_val - min_val + 1e-7)).unsqueeze(0)
to_pred = ((X_another - mean) / std).unsqueeze(0)
#to_pred = X_another.unsqueeze(0)
with torch.no_grad():
    predictions = model(to_pred)
    print(predictions)
    predicted_labels = torch.argmax(predictions, dim=1)
    print("Predicted Labels:", predicted_labels)

tensor([[ -9.6350, -14.2120,  -3.1778,  -4.8483,  -9.2261,  -1.3774,  -4.5930,
           1.1289]])
Predicted Labels: tensor([7])


In [293]:
X_another.shape

torch.Size([184, 6])

In [294]:
X_one.shape

torch.Size([100, 6])

In [295]:
torch.cat([X_another, X_one]).shape

torch.Size([284, 6])

## ONNX Export

In [296]:
X_one.unsqueeze(0).shape

torch.Size([1, 100, 6])

In [134]:
model_path = "./model/gesture_rec_static.onnx"
input_name = "readings"
output_name = "gesture"
input_tensor = torch.randn(1, 100, 6)#X_one.unsqueeze(0)

model.eval()

torch.onnx.export(
    model,
    input_tensor,
    model_path,
    input_names = [input_name],
    output_names = [output_name],
    # dynamic_shapes=({
    #     0: torch.export.Dim.DYNAMIC,
    #     1: torch.export.Dim.DYNAMIC
    # },),
    # dynamic_shapes={
    #     'x': {
    #         0: "batch_size",
    #         1: "seq_len"
    #     }
    # },
    # dynamic_axes={
    #     input_name: {
    #         0: "batch_size",
    #         1: "seq_len"
    #     },
    #     output_name: {
    #         0: "batch_size",
    #         1: "seq_len" 
    #     }
    # },
    dynamo=False,
    external_data=False
)

/home/kojo/tmp/ipykernel_36500/1916155734.py:8: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/home/kojo/Code/730SemesterProject/exploration/imenv/lib/python3.12/site-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset9.py:4463: UserWarning: Exporting a model to ONNX with a batch_size other than 1, with a variable length with LSTM can cause an error when running the ONNX model with a different batch size. Make sure to save the model with a batch size of 1, or define the initial states (h0/c0) as inputs of the model. 
  return _generic_rnn(


> Need to run onnxsim after this:

```bash
onnxsim ./model/gesture_rec_static.onnx ./model/gesture_rec_static_sim.onnx
```